In [ ]:
# 1. install necessary packages
!pip install -q transformers scikit-learn pandas

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import math
import json
import os
import random
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 2. Set the computing device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Current computation device: {device}")

# 3. Load the full dataset
print("Loading the FULL dataset for BERT Baseline training...")
train_df = pd.read_json('/content/train_indexed.jsonl', lines=True)
test_df = pd.read_json('/content/test_indexed.jsonl', lines=True)

train_df['text'] = train_df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)
train_df['rating'] = train_df['rating'].astype(float)
test_df['rating'] = test_df['rating'].astype(float)

# 4. PyTorch Dataset
class AmazonReviewTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=64):
        self.texts = df['text'].values
        self.ratings = df['rating'].values
        self.users = df['user_idx'].values
        self.items = df['item_idx'].values
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        rating = self.ratings[idx]
        user = self.users[idx]
        item = self.items[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'rating': torch.tensor(rating, dtype=torch.float),
            'user_idx': torch.tensor(user, dtype=torch.long),
            'item_idx': torch.tensor(item, dtype=torch.long)
        }

print("Initializing BERT Tokenizer and DataLoaders...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = AmazonReviewTextDataset(train_df, tokenizer)
test_dataset = AmazonReviewTextDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 5. Initialize the BERT model
print("Loading pre-trained BERT model...")
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1)

#  Domain adaptation: freeze the lower layers and unfreeze the last 2 layers
for param in model.bert.parameters():
    param.requires_grad = False
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

model = model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-5)

# 6. Training Phase (Unified Max Epochs = 3)

epochs = 3
print(f"\n Starting Baseline 2 (BERT Content-Based) FULL Training for {epochs} epochs...")

# ----------------- EARLY STOPPING CONFIGURATION -----------------
# Pre-trained language models are highly prone to overfitting, leading to catastrophic forgetting
# Set the patience threshold to patience = 1
patience = 1
best_rmse = float('inf')
patience_counter = 0
# ----------------------------------------------------------------

for epoch in range(epochs):
    model.train()
    total_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ratings = batch['rating'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.squeeze()

        loss = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

        if (batch_idx + 1) % 500 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} | Current Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    # Regression evaluation at the end of each epoch
    model.eval()
    test_preds, test_trues = [], []
    with torch.no_grad():
        for batch_idx, batch in enumerate(test_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ratings = batch['rating']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.squeeze().cpu().numpy()

            if preds.ndim == 0:
                preds = np.expand_dims(preds, axis=0)

            test_preds.extend(preds.tolist())
            test_trues.extend(ratings.tolist())

    rmse = math.sqrt(mean_squared_error(test_trues, test_preds))
    mae = mean_absolute_error(test_trues, test_preds)
    print(f"=== Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Test RMSE: {rmse:.4f} | Test MAE: {mae:.4f} ===")

    # ----------------- EARLY STOPPING CHECK -----------------
    if rmse < best_rmse:
        best_rmse = rmse
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:。
            # Execute a silent break
            break
    # --------------------------------------------------------

# Save Baseline 2 model weights
model_save_path = '/content/baseline2_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f"\n Model weights successfully saved to: {model_save_path}")

# 7. Fairly aligned Top-10 sampled evaluation (Target + 99 Negatives)

print("\n Performing FAIR Sampled Top-10 Evaluation (Target + 99 Negatives)...")

# 1) Build global text profiles for items
item_profiles_df = train_df.groupby('item_idx')['text'].apply(lambda x: ' '.join(x)).reset_index()
item_profiles = dict(zip(item_profiles_df['item_idx'], item_profiles_df['text']))
all_items = set(train_df['item_idx'].unique()).union(set(test_df['item_idx'].unique()))
for i in all_items:
    if i not in item_profiles:
        item_profiles[i] = ""

# 2) Extract user history to avoid overlap with negative samples
train_user_items = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
test_user_items = test_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
all_items_list = list(all_items)

def calculate_ndcg_sampled(recommended_list, interacted_set):
    dcg = sum([1.0 / math.log2(i + 2) for i, item in enumerate(recommended_list) if item in interacted_set])
    idcg = sum([1.0 / math.log2(i + 2) for i in range(min(len(interacted_set), len(recommended_list)))])
    return dcg / idcg if idcg > 0 else 0.0

# Force the same random seed (fully aligned with Baseline 1)
random.seed(42)
test_users_full = list(test_user_items.keys())
evaluate_users_sampled = random.sample(test_users_full, min(1000, len(test_users_full)))

total_precision_samp = 0.0
total_recall_samp = 0.0
total_ndcg_samp = 0.0
valid_users_count = 0

model.eval()
with torch.no_grad():
    for u in tqdm(evaluate_users_sampled, desc="Evaluating Sampled Users"):
        true_items = test_user_items.get(u, set())
        if not true_items:
            continue

        seen_items = train_user_items.get(u, set()).union(true_items)
        negative_items = set()
        while len(negative_items) < 99:
            rand_item = random.choice(all_items_list)
            if rand_item not in seen_items:
                negative_items.add(rand_item)

        candidate_items = list(true_items) + list(negative_items)
        candidate_texts = [item_profiles[i] for i in candidate_items]

        # Use BERT to encode the candidate texts

        encoded_inputs = tokenizer(
            candidate_texts,
            max_length=64,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoded_inputs['input_ids'].to(device)
        attention_mask = encoded_inputs['attention_mask'].to(device)

        # Use the BERT model for inference
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        scores = outputs.logits.squeeze().cpu().numpy()

        item_score_pairs = list(zip(candidate_items, scores))
        item_score_pairs.sort(key=lambda x: x[1], reverse=True)
        top_10_items = [pair[0] for pair in item_score_pairs[:10]]

        hits = len(set(top_10_items) & true_items)
        total_precision_samp += hits / 10.0
        total_recall_samp += hits / len(true_items)
        total_ndcg_samp += calculate_ndcg_sampled(top_10_items, true_items)
        valid_users_count += 1

final_precision_samp = total_precision_samp / valid_users_count
final_recall_samp = total_recall_samp / valid_users_count
final_ndcg_samp = total_ndcg_samp / valid_users_count

print(f"\n Evaluation Complete!")
print(f"Sampled Precision@10: {final_precision_samp:.4f}")
print(f"Sampled Recall@10: {final_recall_samp:.4f}")
print(f"Sampled NDCG@10: {final_ndcg_samp:.4f}")

# 8. Save the regression and ranking results

log_path = '/content/second_experiment_results.json'

baseline2_results = {
    "model_name": "Baseline 2 (BERT Content-Based) - Fair Sampled",
    "RMSE": float(rmse),
    "MAE": float(mae),
    "Precision_10": float(final_precision_samp),
    "Recall_10": float(final_recall_samp),
    "NDCG_10": float(final_ndcg_samp),
    "epochs": epochs,
    "eval_users": valid_users_count,
    "Notes": "Evaluated via Target+99 Negatives on 1000 sampled users using Item Profiling"
}

if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        all_results = json.load(f)
    # Clean up any old Baseline 2 data
    all_results = [res for res in all_results if "Baseline 2" not in res.get("model_name", "")]
else:
    all_results = []

all_results.append(baseline2_results)

with open(log_path, 'w') as f:
    json.dump(all_results, f, indent=4)

print("\n Baseline 2 results successfully updated in second_experiment_results.json!")

Current computation device: cuda
Loading the FULL dataset for BERT Baseline training...
Initializing BERT Tokenizer and DataLoaders...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading pre-trained BERT model...


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



 Starting Baseline 2 (BERT Content-Based) FULL Training for 3 epochs...
  Epoch 1 | Batch 500/5430 | Current Loss: 0.6608
  Epoch 1 | Batch 1000/5430 | Current Loss: 0.7234
  Epoch 1 | Batch 1500/5430 | Current Loss: 0.8001
  Epoch 1 | Batch 2000/5430 | Current Loss: 0.4139
  Epoch 1 | Batch 2500/5430 | Current Loss: 0.4327
  Epoch 1 | Batch 3000/5430 | Current Loss: 0.1921
  Epoch 1 | Batch 3500/5430 | Current Loss: 0.6115
  Epoch 1 | Batch 4000/5430 | Current Loss: 0.5186
  Epoch 1 | Batch 4500/5430 | Current Loss: 0.5266
  Epoch 1 | Batch 5000/5430 | Current Loss: 1.1394
=== Epoch 1 Summary | Train Loss: 0.7684 | Test RMSE: 0.7725 | Test MAE: 0.4298 ===
  Epoch 2 | Batch 500/5430 | Current Loss: 0.1849
  Epoch 2 | Batch 1000/5430 | Current Loss: 0.8446
  Epoch 2 | Batch 1500/5430 | Current Loss: 0.2662
  Epoch 2 | Batch 2000/5430 | Current Loss: 0.1598
  Epoch 2 | Batch 2500/5430 | Current Loss: 1.1807
  Epoch 2 | Batch 3000/5430 | Current Loss: 0.6075
  Epoch 2 | Batch 3500/5430 |

Evaluating Sampled Users: 100%|██████████| 1000/1000 [09:41<00:00,  1.72it/s]


 Evaluation Complete!
Sampled Precision@10: 0.0216
Sampled Recall@10: 0.1062
Sampled NDCG@10: 0.0601

 Baseline 2 results successfully updated in second_experiment_results.json!
